# Day 05 — Feature Engineering & Data Encoding

This notebook converts selected categorical Titanic features into numerical representations suitable for machine-learning workflows. It demonstrates binary mapping for `sex`, one-hot encoding for `embarked`, concatenation with the remaining features, and a sanity check of the transformed data.


In [ ]:
import pandas as pd
import seaborn as sns

# Load the Titanic dataset
df = sns.load_dataset("titanic")

print("Original data shape:", df.shape)
display(df.head())


## 1. Binary Encoding — `sex`

The `sex` feature has two categories, so a direct mapping is sufficient: `male` becomes `1` and `female` becomes `0`. The mapping is explicit and keeps the transformation easy to inspect. The numeric values are category codes; they should not be interpreted as a measured quantity or a ranking.


In [ ]:
# Map the two categories to a binary numeric representation
df["sex"] = df["sex"].map({"male": 1, "female": 0})

print("Sex values after binary mapping:")
print(df["sex"].value_counts(dropna=False))


## 2. One-Hot Encoding — `embarked`

The `embarked` feature represents nominal port categories. Assigning arbitrary integers (for example, 0, 1, and 2) could falsely suggest an order or distance between ports. One-hot encoding instead creates indicator columns, one for each represented category.

`drop_first=True` removes one indicator column and treats it as the reference category. With an intercept in a linear model, retaining every dummy column can create perfect linear dependence (the dummy-variable trap). Dropping one column avoids that exact multicollinearity in this common setup. The reference category is represented when all retained dummy indicators are zero.


In [ ]:
# Create one-hot columns for embarked and drop the first category as reference
embarked_encoded = pd.get_dummies(
    df["embarked"],
    prefix="embarked",
    drop_first=True
)

print("Generated embarked columns:")
print(embarked_encoded.columns.tolist())
display(embarked_encoded.head())


## 3. Concatenate Encoded Features & Validate

After generating the dummy variables, I remove the original text-based `embarked` column and concatenate the encoded columns back to the remaining DataFrame along `axis=1` (columns). A small sample and structural checks make it easier to confirm that the transformation produced the expected schema.


In [ ]:
# Remove the original categorical column before adding its encoded representation
df = df.drop(columns=["embarked"])

# Add the one-hot encoded columns side by side
df = pd.concat([df, embarked_encoded], axis=1)

# Sanity checks for the transformed dataset
print("Encoded data shape:", df.shape)
print("\\nEncoded columns:")
print(df.columns.tolist())
print("\\nEncoded sample data:")
display(df.head())
print("\\nData types:")
print(df.dtypes)


## Engineering Takeaways

- Binary mapping is appropriate for a genuinely two-category feature when the mapping is explicit and documented.
- One-hot encoding is a safer representation for nominal categories because it does not invent an ordinal relationship.
- `drop_first=True` removes a reference dummy and avoids perfect multicollinearity when a model includes an intercept.
- The encoded columns are concatenated with `pd.concat(..., axis=1)`, and the resulting sample, schema, and data types are inspected.

**Production note:** Learn category mappings and preprocessing parameters from the training split only, then apply the same fitted transformation to validation and test data. In a production machine-learning pipeline, `OneHotEncoder` inside a Scikit-learn `ColumnTransformer` is generally preferable because it helps keep preprocessing consistent and leakage-safe.
